# Train YOLOv8 on Retail Products Classification (Kaggle)

Kaggle mounts the competition at `/kaggle/input`, so there is no API token, no `kaggle.json`
and no 401 to debug — the data is simply there.

**Before running, in the right-hand panel:**

1. **Input → Add Input** → search `retail-products-classification` → **Add**. If it will not
   add, open the competition page and accept the rules; if the page offers no accept button,
   the InClass competition is closed and this route will not work either.
2. **Settings → Accelerator → GPU T4 x2** (or P100).
3. **Settings → Internet → On** (needed to clone the repo and pip install).

Trains `yolov8n-cls`, since this dataset labels whole images across 21 categories and ships
no bounding boxes.

## 1. Confirm the data is mounted
Lists the labels and, more importantly, **where the images are** - the prep step needs
that path. If no images appear, read the note this cell prints before running anything else.


In [ ]:
import os

INPUT_DIR = "/kaggle/input"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

if not os.path.isdir(INPUT_DIR) or not os.listdir(INPUT_DIR):
    raise SystemExit("Nothing mounted. Add Input > retail-products-classification, then re-run.")

# Every directory that directly holds images, with a count. The competition nests them in a
# way that is not documented anywhere, and the prep script has to find them - so see them here
# first rather than discovering the layout from a failure two cells later.
image_dirs = {}
csv_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    dirs[:] = [d for d in dirs if d not in {"__MACOSX", ".ipynb_checkpoints"}]
    images = [f for f in files if f.lower().endswith(IMAGE_EXTENSIONS)]
    if images:
        image_dirs[root] = (len(images), sorted(images)[:2])
    csv_files += [os.path.join(root, f) for f in files if f.lower().endswith((".csv", ".zip"))]

print("csv / zip files:")
for path in sorted(csv_files)[:10]:
    print(f"  {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")

total = sum(count for count, _ in image_dirs.values())
print(f"\nimage directories: {len(image_dirs)} holding {total:,} images")
for path, (count, sample) in sorted(image_dirs.items())[:10]:
    print(f"  {path}  ({count:,} images, e.g. {sample})")

if not image_dirs:
    print(
        "\nNo images are mounted. The labels alone cannot train anything: this competition "
        "publishes its images separately, so find the matching image dataset under Add Input, "
        "then pass --images-dir <that path> to the training cells below."
    )


## 2. Check the GPU

In [ ]:
import subprocess

import torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip() or "no GPU")

assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2, then re-run."
print(f"torch {torch.__version__} | cuda {torch.version.cuda}")

## 3. Get the training scripts

Cloned into `/kaggle/working`, the only writable location — the scripts write the built
dataset and the model there.

Needs **Internet: On**, and the branch below must actually be pushed to GitHub. If the clone
fails or the scripts turn out to predate `--task`, use the upload fallback in the next cell.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/weshallsah/yolo.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/yolo"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

# The classification pipeline is what this notebook drives, so check it is actually present
# rather than failing later with an opaque argparse error.
help_text = subprocess.run(
    ["python", "backend/scripts/train_on_retail.py", "--help"],
    capture_output=True, text=True,
).stdout
if "--task" not in help_text:
    raise SystemExit(
        "This checkout predates the classification pipeline: train_on_retail.py has no "
        "--task flag. Push the latest commits to GitHub and re-run this cell."
    )
print("\nclassification pipeline present")

### Fallback if the repo is not pushed yet

Skip this if cell 3 succeeded. Otherwise: zip `backend/scripts/` locally, upload it as a
Kaggle Dataset (**Add Input → Upload**), and point `SCRIPTS_SRC` at it.

In [ ]:
import os
import shutil

SCRIPTS_SRC = None  # e.g. "/kaggle/input/yolo-scripts"

if SCRIPTS_SRC:
    destination = "/kaggle/working/yolo/backend/scripts"
    os.makedirs(destination, exist_ok=True)
    for name in os.listdir(SCRIPTS_SRC):
        if name.endswith(".py"):
            shutil.copy(os.path.join(SCRIPTS_SRC, name), destination)
            print("copied", name)
    os.chdir("/kaggle/working/yolo")
else:
    print("skipped - using the cloned repo")

## 4. Install Ultralytics
Kaggle ships a version that lags, so pin it to match the repo's requirements.

In [ ]:
!pip install -q "ultralytics==8.4.150"

import ultralytics

ultralytics.checks()

## 5. Smoke run

2,000 images, one epoch. Catches a column-name surprise or an OOM in ~2 minutes rather than
an hour into the real run.

`--workers 2` because Kaggle's shared-memory limit deadlocks higher worker counts.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --max-images 2000 \
    --epochs 1 \
    --batch 64 \
    --workers 2 \
    --cache none

## 6. Full training run

`--force` is required — cell 5 left a capped 2,000-image build behind, and without it that
subset is silently reused.

Watch `metrics/accuracy_top1`. Raise `--epochs` if it is still climbing at the end; add
`--max-per-class 3000` if the prep step reports steep class imbalance.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --force \
    --epochs 30 \
    --imgsz 224 \
    --batch 64 \
    --workers 2 \
    --cache disk

## 7. Keep the weights

Everything under `/kaggle/working` is saved as notebook output when the session ends, but the
built dataset is tens of thousands of files that bloat it for no reason. This copies just the
run directory to the top level and deletes the dataset.

Download afterwards from the notebook's **Output** tab.

In [ ]:
import shutil

RUN_DIR = "/kaggle/working/yolo/backend/models/retail_yolo_products_cls"
KEEP_DIR = "/kaggle/working/retail_yolo_products_cls"

shutil.copytree(RUN_DIR, KEEP_DIR, dirs_exist_ok=True)
shutil.rmtree("/kaggle/working/yolo/backend/training_data", ignore_errors=True)

for root, _, files in os.walk(KEEP_DIR):
    for name in sorted(files):
        path = os.path.join(root, name)
        print(f"{os.path.getsize(path) / 1e6:8.1f} MB  {path}")

## 8. Sanity-check the classifier
Predicts held-out validation images. Run this before cell 7 deletes the dataset, or re-point
it at any image you like.

In [ ]:
import glob
import os

from ultralytics import YOLO

model = YOLO(os.path.join(KEEP_DIR, "weights", "best.pt"))
samples = sorted(glob.glob(
    "/kaggle/working/yolo/backend/training_data/retail/dataset_products_cls/val/*/*"
))[:10]

for result in model.predict(samples, verbose=False):
    truth = os.path.basename(os.path.dirname(result.path))
    predicted = result.names[result.probs.top1]
    mark = "ok " if truth == predicted else "MISS"
    print(f"{mark} true={truth:28s} pred={predicted:28s} conf={result.probs.top1conf:.2f}")

## Serving it

Download `best.pt` from the Output tab into `backend/models/retail_yolo_products_cls/weights/`,
then:

```bash
# backend/.env
APP_YOLO_WEIGHTS_PATH=models/retail_yolo_products_cls/weights/best.pt
APP_MODEL_TASK=classify
```

`APP_MODEL_TASK=classify` is required: a `-cls` checkpoint exposes `result.probs` and no
`result.boxes`, so the default detection path would return an empty list for every image.